# Chat session -> turns

A chat function between a **human** (a diabetes patient) and an **agent** (a lifestyle coach),
matching the conversational-coaching setup already used by `events_from_chat`
(`llm_event_triples_openai_pydantic.py`, `prompts.py`).

Every exchange is converted into a flat list of **turns**. Each turn is a dict:

```python
{
    "chat": <chat identifier>,      # which conversation this turn belongs to
    "human": <human's name>,        # the patient's name, same for every turn in the chat
    "date": <date the chat took place>,
    "turn": <turn identifier>,      # 1, 2, 3, ... in speaking order within this chat
    "speaker": <"human" name or "agent">,
    "utterance": <the text that was said>,
}
```

This is exactly the shape `annotate_all_turns_in_conversation` expects as `input['turns']` items
(see `llm_event_triples_openai_pydantic.py`), so the output of this notebook can be fed straight
into that annotation pipeline.

`ChatSession` and its helpers live in **[`chat_sessions.py`](chat_sessions.py)** (next to this
notebook) rather than in a cell here, so this stays a short usage notebook. Two ways to produce
a conversation are provided:

- **`run_gui(ChatSession(...))`** — a live back-and-forth in its own window
  (**[`kg_chat_gui.py`](kg_chat_gui.py)**, shared with `kg_chat_session.ipynb`): you type as the
  human into that window's own entry box, the agent (an OpenAI model) replies, turn by turn, both
  shown scrolling in the same window. Quitting saves the turns and a statistics summary as JSON.
- **`simulate_chat()`** — a scripted, non-interactive conversation (a fixed list of human lines),
  useful for testing or demos without typing anything, and runnable with a mock agent that needs
  no API key at all.

For the knowledge-graph-populating version of this same idea, see the companion notebook
**[`kg_chat_session.ipynb`](kg_chat_session.ipynb)** (`KgChatSession`, also in `chat_sessions.py`).


## Setup

`ChatSession` reads `OPENAI_API_KEY` lazily -- only the first time an OpenAI-backed agent reply
is actually needed -- so this notebook still runs end-to-end with the mock agent even if no key
is set (see `chat_sessions._load_key()` / `_openai_client()`).


In [1]:
from chat_sessions import ChatSession, mock_agent, simulate_chat, save_turns


## Run a live chat

Opens the conversation in its own window (`kg_chat_gui.py`'s `ChatWindow`, shared with
`kg_chat_session.ipynb`): the transcript scrolls in that one window, and you type your next line
into the entry box built into the bottom of that *same* window -- no separate popup, and no
notebook `input()` prompt (which doesn't work reliably in every Jupyter frontend). Every agent
turn is labeled **[LLM]** -- a plain `ChatSession` only ever has this one kind of reply; the
**[KG]** label and the gap-sensitivity slider are specific to `KgChatSession`, see
`kg_chat_session.ipynb`.

**To stop:** click **Quit**, or type "quit"/"bye"/... The window closes and the cell finishes,
and the conversation plus a statistics summary are written to two timestamped JSON files in the
notebook's directory (`chat<chat>_turns_<stamp>.json` / `chat<chat>_stats_<stamp>.json`) -- their
paths are printed below the cell. `run_gui()` then returns `session.turns`, the same shape
`run_interactive()` returned, so everything below still works; pass `save_dir=None` to skip the
automatic save. Requires `OPENAI_API_KEY`, since it uses `openai_agent()` by default.


In [ ]:
from kg_chat_gui import run_gui

session = ChatSession(chat=1, human="Mehmet")
turns = run_gui(session)
turns


## Scripted / offline demo

`simulate_chat()` runs the same turn-by-turn exchange non-interactively from a fixed list of
human lines -- handy for testing the turn format, or for a repeatable demo. `mock_agent` needs
no API key at all; swap in `chat_sessions.openai_agent()` (or omit `agent_fn` to get it by
default) to use a real model instead.


In [ ]:
demo_turns = simulate_chat(
    chat=254,
    human="Mehmet",
    date="2013,Jan,31",
    agent_fn=mock_agent,
    human_utterances=[
        "I've been reading about how different diets can impact my blood sugar levels. "
        "Recently, I've heard about the Mediterranean diet being beneficial. Do you think "
        "it's suitable for me?",
        "Not entirely, but I've been incorporating more olive oil and fish into my meals. "
        "Is there anything specific I should avoid or be cautious about?",
    ],
)
demo_turns


## Inspect and save

`turns` is already the flat list of `{chat, human, date, turn, speaker, utterance}` dicts. A
DataFrame view makes it easy to eyeball, and `save_turns()` writes it out as JSON for downstream
use (e.g. as `input['turns']` to `annotate_all_turns_in_conversation`, via `session.as_conversation()`).


In [ ]:
import pandas as pd

pd.DataFrame(turns)


In [ ]:
save_turns(turns, "turns.json")
